In [ ]:
import numpy as np

def J1solve(Q, Anglediff, i, j):  
    if i == j:
        return (-Q[i] - B[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * Vsp[j] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff)))

def J2solve(P, Anglediff, i, j):
    if i == j:
        return (P[i] + G[i,i] * (Vsp[i]**2))
    else:
        return (-Vsp[i] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff)))

def J3solve(P, Anglediff, i, j):
    if i == j:
        return (P[i] - G[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * Vsp[j] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff)))

def J4solve(Q, Anglediff, i, j):
    if i == j:
        return (Q[i] - B[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff)))
    
    
def clean_slack(J1, J2, J3, J4, mismatch):
    '''
    The final Jacobian should be 8x8
    where:
    dp/detheta=J1=[dp0/dtheta0, dp0/dtheta1, dp1/dtheta0, dp1/dtheta1]
    dp/dV=J2=[dp0/dV0, dp0/dV1, dp1/dV0, dp1/dV1]
    dq/detheta=J3=[dq0/dtheta0, dq0/dtheta1, dq1/dtheta0, dq1/dtheta1]
    dq/dV=J4=[dq0/dV0, dq0/dV1, dq1/dV0, dq1/dV1]
    We need to remove all elements except the last one which is related to pq load or
    [[0,0,0,1], [0,0,0,1], [0,0,0,1], [0,0,0,1]], where 0 is slack bus and 1 is pq load
    we only need filter the first 3 elements of J1, J2, J3, J4 to get the jacobians related to pq
    '''
    J1r=J1[1:, 1:]
    J2r=J2[1:, 1:]
    J3r=J3[1:, 1:]
    J4r=J4[1:, 1:]
    J=np.block([[J1r, J2r], [J3r, J4r]])
    #filter out the first element of mismatch.
    mismatch_reduced = np.concatenate([
        mismatch[0][1:],  
        mismatch[1][1:]  
    ])
    
    return J, mismatch_reduced

#Initial condition
Vsp = np.array([1.0, 1.0])
Angsp = np.array([0.0, 0.0])
Psp = np.array([0, -200e6/100e6])
Qsp = np.array([0, -100e6/100e6])
busamount = 2;

#Simple line impedance
Z = 0.1j
Y=1/(Z)
Ymatrix = np.array([[Y, -Y], [-Y, Y]])

#G and B matrices, G should be 0 since no resistance in the line
G = np.real(Ymatrix)
B = np.imag(Ymatrix)
dP = np.zeros(len(Psp))
dQ = np.zeros(len(Qsp))

#Newton-Raphson iteration
for iter in range(20):
    P = np.zeros(len(Psp))
    Q = np.zeros(len(Qsp))
    J1 = np.zeros((len(Psp), len(Psp)))
    J2 = np.zeros((len(Psp), len(Psp)))
    J3 = np.zeros((len(Psp), len(Psp)))
    J4 = np.zeros((len(Psp), len(Psp)))
    #calculate P, Q, and Jacobian
    for i in range(busamount):
        for j in range(busamount):
            Anglediff = Angsp[i] - Angsp[j]
            P[i] += Vsp[i] * Vsp[j] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff))
            Q[i] += Vsp[i] * Vsp[j] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff))
        for k in range(busamount):
            Anglediff = Angsp[i] - Angsp[k]
            J1[i, k] += J1solve(Q, Anglediff, i, k)
            J2[i, k] += J2solve(P, Anglediff, i, k)
            J3[i, k] += J3solve(P, Anglediff, i, k)
            J4[i, k] += J4solve(Q, Anglediff, i, k)
    dP = Psp - P
    dQ = Qsp - Q
    mismatch = np.array([dP, dQ])

    #filter out the first element of mismatch (slack bus)
    J, mismatch = clean_slack(J1, J2, J3, J4, mismatch)

    # Check convergence
    if np.max(np.abs(mismatch)) < 1e-6:
        print(f"Converged in {iter+1} iterations")
        break
    #Linear system solve for X
    X = np.linalg.solve(J, mismatch)
    
    delta_theta = X[0].item()
    delta_V = X[1].item()

    #Voltage is V/V, angle is rad, multiply by V for updating. 
    for i in range(1, len(Vsp)):
        Vsp[i] += delta_V*Vsp[i]
        Angsp[i] += delta_theta

print("Voltage of Load Bus 2: " + str(Vsp[1]) + " pu")
print("Angle of Load Bus 2: " + str(Angsp[1]*180/np.pi) + " deg")


In [ ]:
!sudo apt-get update
!sudo apt-get upgrade
!pip install numpy==2.2.3


### Homework 4
Consider the two-bus power system shown in the figure above. The system consists of:
A slack bus (Bus 1) with a voltage magnitude of 1 per unit (p.u.) and a zero voltage angle.
A load bus (Bus 2), where active power (P) and reactive power (Q) are specified.
A transmission line connecting Bus 1 and Bus 2, modeled as an impedance of Z=j0.1 p.u.
System Base Power: Sbase=100 MVA
Tasks:

1. Formulate the power flow equations using the Newton-Raphson method for this system.
2. Determine the unknown variables:
    > Voltage magnitude and angle at Bus 2.
3. Solve the power flow problem iteratively using the Newton-Raphson method.
4. Discuss the convergence of the NR method for this simple two-bus system.
5. Validate your results using either PowerWorld or MATPOWER.

I've converted my previous code and Matlab examples to Python for easy debugging during the exam, this homework was done in Collab Notebook.
The Newton-Raphson method for a 2 bus system is the following:
